# Component 3: Visual Oracle Evaluator (Raw Global vs. SAHI Multi-Slice Slicing)
### Multimodal Oracle Verification Across Real Android Bug Screens & High-Resolution Viewports
**Student:** M.L. Booso (IT23452916) | **Component:** C3 Local Vision-Semantic Oracle | **Module:** IT4010 / PP1 Defense

---

### Core Research Motivation & Technical Rationale
1. **The Downscaling Blindspot:** Modern Android displays have tall aspect ratios ($20:9$, e.g. $1080 \times 2400$). Downsampling an entire high-resolution screen to a single standard thumbnail blurs fine visual cues—destroying subtle button overlaps, clipped descenders, and micro-text collisions.
2. **SAHI Slicing (Slicing Aided Hyper Inference):** Decomposes the viewport into a coverage pyramid (1 Global Context View + 3 Overlapping Vertical Slices with ~25% overlap). Each tile receives full VLM spatial resolution, catching subtle local defects.
3. **AST View Hierarchy Pruning:** Android accessibility XML/JSON dumps often exceed 15,000 characters and contain hundreds of non-actionable layout wrappers. Our deterministic AST pruner strips invisible nodes, reducing tokens by ~78.4% and injecting structured element context into the prompt.
4. **Structured Constrained Prompting:** Enforces a strict JSON contract, preventing conversational hallucinations and ensuring deterministic test synthesis for Component 4.
5. **Real Bug Screen Verification:** Evaluates raw inspection first, followed by SAHI multi-slice inspection across real benchmark screens (OwlEye, Rico mutations, clean baselines) to show actual model results without relying on hardcoded expected labels.


## 1. Environment & Multi-Root Path Resolution
Verifies hardware environment, CUDA acceleration on NVIDIA RTX 3050 Laptop GPU (4GB VRAM), and multi-root workspace path resolution.


In [ ]:
# Cell 1: Hardware & Environment Telemetry
import os
import sys
import time
import json
import math
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Resilient multi-root path resolver
def resolve_path(rel_candidate):
    cand = Path(rel_candidate)
    if cand.is_absolute() and cand.exists():
        return cand
    search_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("..").resolve(),
        Path("../..").resolve(),
        Path(r"D:/university/Resesarch"),
        Path(r"D:/university/Resesarch/J26-SE-334"),
        Path(r"D:/university/Resesarch/local-vision-oracle"),
    ]
    for root in search_roots:
        target = (root / cand).resolve()
        if target.exists():
            return target
    return (Path(r"D:/university/Resesarch") / cand).resolve()

# Add workspace roots to sys.path
for root_name in ["J26-SE-334", "local-vision-oracle"]:
    ws = resolve_path(root_name)
    if ws.exists() and str(ws) not in sys.path:
        sys.path.insert(0, str(ws))

print("=== HARDWARE & ENVIRONMENT TELEMETRY ===")
print(f"[*] Python Executable : {sys.executable}")
print(f"[*] PyTorch Version   : {torch.__version__}")
print(f"[*] CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Compute Device    : {torch.cuda.get_device_name(0)}")
    vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
    print(f"[*] Dedicated VRAM    : {vram_mb:.0f} MB (RTX 3050 Laptop GPU verified)")
else:
    print("[!] Running in CPU mode")


## 2. Model Loading (4-Bit NF4 LoRA on 4GB VRAM)
Loads the base `Qwen2-VL-2B-Instruct` model compressed with **4-bit NF4 quantization** and attaches our fine-tuned **boosted recall LoRA adapter** (`c3_lora_boosted_recall`, Step 246).
Fits entirely within **~1.57 GB of VRAM**, safely executing offline on consumer hardware with zero network traffic.


In [ ]:
# Cell 2: 4-Bit NF4 Quantization & LoRA Adapter Loading
import gc
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BASE_MODEL_PATH = resolve_path("models/Qwen2-VL-2B-Instruct")
if not BASE_MODEL_PATH.exists():
    BASE_MODEL_PATH = resolve_path("RD/models/Qwen2-VL-2B-Instruct")

# Load verified boosted recall weights
LORA_PATH = resolve_path("models/c3_lora_boosted_recall")
if not (LORA_PATH / "adapter_model.safetensors").exists():
    LORA_PATH = resolve_path("results/c3_lora_boosted_recall")

print("=== INITIALIZING LOCAL VISION ORACLE (4-BIT NF4) ===")
print(f"[*] Base Model Path : {BASE_MODEL_PATH}")
print(f"[*] LoRA Weights    : {LORA_PATH}")
assert BASE_MODEL_PATH.exists(), f"Base model missing at {BASE_MODEL_PATH}"
assert (LORA_PATH / "adapter_model.safetensors").exists(), f"LoRA weights missing at {LORA_PATH}"

# Configure 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)

print("[1/2] Loading processor and base model into GPU...")
t_load_start = time.time()
processor = AutoProcessor.from_pretrained(str(BASE_MODEL_PATH))

# Use explicit GPU device mapping to prevent meta tensor offloading
device_map = {"": 0} if torch.cuda.is_available() else "auto"
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    str(BASE_MODEL_PATH),
    quantization_config=bnb_config,
    device_map=device_map,
)

print("[2/2] Attaching fine-tuned LoRA domain adapter...")
model = PeftModel.from_pretrained(base_model, str(LORA_PATH))
model.eval()

device = next(model.parameters()).device
load_sec = time.time() - t_load_start
vram_mb = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
print(f"[+] Model ready on {device} (loaded in {load_sec:.1f}s, VRAM allocated: {vram_mb:.1f} MB)")


## 3. Technique 1: AST View Hierarchy Pruning
Static XML dumps from `uiautomator` or Rico JSON hierarchies are bloated with layout containers (`FrameLayout`, `LinearLayout`, `ViewGroup`).
Our deterministic AST pruner (`HierarchyPruner`) traverses the tree, removes non-interactive and non-text wrappers, normalizes bounding boxes to $[0, 1000]$, and reduces token size by **~78.4%**.


In [ ]:
# Cell 3: AST View Hierarchy Pruning Demonstration
from vision_oracle.local_vision_oracle.pruner import HierarchyPruner, PrunedNode

# Sample raw Android View Hierarchy XML (representing a checkout screen with button overlap)
raw_xml_sample = '''<?xml version='1.0' encoding='UTF-8' standalone='yes' ?>
<hierarchy rotation="0">
  <node index="0" text="" resource-id="" class="android.widget.FrameLayout" package="com.solvance.android" bounds="[0,0][1080,1920]" checkable="false" checked="false" clickable="false" enabled="true" focusable="false" focused="false" scrollable="false" long-clickable="false" password="false" selected="false">
    <node index="0" text="" resource-id="" class="android.widget.LinearLayout" package="com.solvance.android" bounds="[0,0][1080,1920]" checkable="false" checked="false" clickable="false" enabled="true" focusable="false" focused="false" scrollable="false" long-clickable="false" password="false" selected="false">
      <node index="0" text="" resource-id="com.solvance.android:id/action_bar" class="android.view.ViewGroup" package="com.solvance.android" bounds="[0,72][1080,240]" checkable="false" checked="false" clickable="false" enabled="true" focusable="false" focused="false" scrollable="false" long-clickable="false" password="false" selected="false">
        <node index="0" text="ShopEase Checkout" resource-id="com.solvance.android:id/title" class="android.widget.TextView" package="com.solvance.android" bounds="48,110,600,202" checkable="false" checked="false" clickable="false" enabled="true" focusable="false" focused="false" scrollable="false" long-clickable="false" password="false" selected="false" />
      </node>
      <node index="1" text="" resource-id="com.solvance.android:id/content_frame" class="android.widget.FrameLayout" package="com.solvance.android" bounds="[0,240][1080,1776]" checkable="false" checked="false" clickable="false" enabled="true" focusable="false" focused="false" scrollable="false" long-clickable="false" password="false" selected="false">
        <node index="0" text="" resource-id="com.solvance.android:id/payment_container" class="android.widget.RelativeLayout" package="com.solvance.android" bounds="[60,800][1020,1500]" checkable="false" checked="false" clickable="false" enabled="true" focusable="false" focused="false" scrollable="false" long-clickable="false" password="false" selected="false">
          <node index="0" text="Enter Promo Code" resource-id="com.solvance.android:id/promo_input" class="android.widget.EditText" package="com.solvance.android" bounds="[80,950][1000,1100]" checkable="false" checked="false" clickable="true" enabled="true" focusable="true" focused="false" scrollable="false" long-clickable="true" password="false" selected="false" />
          <node index="1" text="Apply Discount" resource-id="com.solvance.android:id/apply_btn" class="android.widget.Button" package="com.solvance.android" bounds="[700,920][1010,1120]" checkable="false" checked="false" clickable="true" enabled="true" focusable="true" focused="false" scrollable="false" long-clickable="false" password="false" selected="false" />
          <node index="2" text="Proceed to Pay ($49.99)" resource-id="com.solvance.android:id/pay_btn" class="android.widget.Button" package="com.solvance.android" bounds="[80,1320][1000,1470]" checkable="false" checked="false" clickable="true" enabled="true" focusable="true" focused="false" scrollable="false" long-clickable="false" password="false" selected="false" />
        </node>
      </node>
    </node>
  </node>
</hierarchy>'''

pruner = HierarchyPruner(screen_width=1080, screen_height=1920)
pruned_nodes = pruner.prune_xml(raw_xml_sample)
compact_context = pruner.to_prompt_string(pruned_nodes) if hasattr(pruner, "to_prompt_string") else "\n".join(n.to_prompt_string() for n in pruned_nodes)

raw_len = len(raw_xml_sample)
pruned_len = len(compact_context)
reduction_pct = (1.0 - (pruned_len / raw_len)) * 100.0

print("=== AST HIERARCHY PRUNING BENCHMARK ===")
print(f"[*] Raw XML Length       : {raw_len:,} characters")
print(f"[*] Pruned Context Length: {pruned_len:,} characters")
print(f"[*] Token Reduction      : {reduction_pct:.2f}% pruned")
print(f"[*] Retained Elements    : {len(pruned_nodes)} salient interactive/textual nodes\n")
print("--- COMPACT PRUNED DOM CONTEXT (PROMPT INJECTION) ---")
print(compact_context)


## 4. Technique 2: Optimized Image Preprocessing & Aspect-Ratio Preservation
Mobile screens have extreme $20:9$ portrait aspect ratios. Naive square resizing causes severe horizontal stretching that distorts text glyphs and layout spacing.
Our pipeline maintains native aspect ratios and enforces strict token-budget caps per image tile.


In [ ]:
# Cell 4: Aspect-Ratio Preserving Preprocessing
def preprocess_view_image(img, max_dim=(720, 1280)):
    """Scales image preserving exact aspect ratio without geometric distortion."""
    disp = img.copy()
    disp.thumbnail(max_dim, Image.Resampling.LANCZOS)
    return disp

# Test image aspect ratio preservation
test_img = Image.new("RGB", (1080, 2400), color=(240, 243, 246))
proc_img = preprocess_view_image(test_img)

print("=== IMAGE PREPROCESSING GEOMETRY ===")
print(f"[*] Original Physical Dimensions : {test_img.size} (Aspect: {test_img.size[1]/test_img.size[0]:.2f}:1)")
print(f"[*] Preprocessed Dimensions      : {proc_img.size} (Aspect: {proc_img.size[1]/proc_img.size[0]:.2f}:1)")
assert abs((test_img.size[1]/test_img.size[0]) - (proc_img.size[1]/proc_img.size[0])) < 0.05, "Aspect ratio was distorted!"
print("[+] Geometric aspect ratio perfectly preserved for VLM visual encoder.")


## 5. Technique 3: SAHI Spatial Slicing & Inverse Coordinate Mapping
`ViewPlanner` slices tall mobile screens into an overlapping tile pyramid:
- **Global View:** $0\% - 100\%$ full screen context.
- **Slice 0:** Top viewport ($0\% - 40\%$).
- **Slice 1:** Mid-upper viewport ($30\% - 70\%$).
- **Slice 2:** Mid-lower viewport ($60\% - 100\%$).

Bounding boxes detected inside tiles are projected back to global physical screen coordinates via `CoordinateMapper`.


In [ ]:
# Cell 5: SAHI View Planning & Interactive Grid Visualizer
from vision_oracle.local_vision_oracle.pipeline import ViewPlanner, CoordinateMapper, OverlapDeduplicator, CoveragePolicy

views = ViewPlanner.plan_views(1080, 2400, policy=CoveragePolicy.GLOBAL_PLUS_TILES_V1)
print(f"=== SAHI VIEW PLANNING DECOMPOSITION ({len(views)} Views) ===")
for v in views:
    print(f"  -> View ID: {v.view_id:8s} | Global: {str(v.is_global):5s} | Crop: {v.crop_rect}")

# Render the multi-view slicing grid
fig, axes = plt.subplots(1, 4, figsize=(14, 5), dpi=100)
sample_mock = Image.new("RGB", (1080, 2400), color=(245, 247, 250))
draw = ImageDraw.Draw(sample_mock)
for y in range(200, 2400, 300):
    draw.rectangle([100, y, 980, y+140], fill=(220, 225, 230), outline=(180, 190, 200), width=3)

colors = ["#2b6cb0", "#38a169", "#dd6b20", "#805ad5"]
for idx, (ax, view) in enumerate(zip(axes, views)):
    cx, cy, cw, ch = view.crop_rect
    crop_img = sample_mock if view.is_global else sample_mock.crop((cx, cy, cx+cw, cy+ch))
    ax.imshow(crop_img)
    ax.set_title(f"{view.view_id}\n({cw}x{ch} px)", fontsize=11, fontweight="bold", color=colors[idx])
    ax.axis("off")
    rect = patches.Rectangle((0, 0), crop_img.width-1, crop_img.height-1, linewidth=3, edgecolor=colors[idx], facecolor="none")
    ax.add_patch(rect)

plt.suptitle("SAHI Multi-View Decomposition: 1 Global View + 3 Overlapping Vertical Slices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Technique 4: Structured Constrained Prompting & Strict Validation
`FindingParser` enforces strict schema validation:
- Requires valid JSON object with `has_defect` boolean and `defect_type` enum.
- Rejects string booleans (`"false"`), invalid/reversed bounding boxes, or NaN confidences.
- Inconclusive verdicts are preserved (never defaulting to clean!).


In [ ]:
# Cell 6: Structured Constrained Prompt & FindingParser Validation
from vision_oracle.local_vision_oracle.pipeline import FindingParser

system_prompt = (
    "You are an automated visual testing oracle for mobile GUI defect detection.\n"
    "Identify supported defects: 'component_overlap', 'text_truncation', or 'none'.\n"
    "Return STRICT JSON ONLY:\n"
    "{\n"
    '  "has_defect": bool,\n'
    '  "defect_type": "component_overlap" | "text_truncation" | "none",\n'
    '  "confidence": float,\n'
    '  "bounding_box": [ymin, xmin, ymax, xmax] | null,\n'
    '  "affected_element_id": str | null,\n'
    '  "explanation": "concise technical description"\n'
    "}"
)

# Test parser against malformed and valid inputs
sample_view = views[0]
valid_json = '{"has_defect": true, "defect_type": "component_overlap", "confidence": 0.94, "bounding_box": [100, 150, 400, 850], "explanation": "Button overlaps text."}'
malformed_json = '{"has_defect": "true", "bounding_box": [500, 100, 200, 300]}' # string bool + reversed box

findings_valid, errors_valid = FindingParser.parse_findings(valid_json, sample_view, 1080, 1920)
findings_malformed, errors_malformed = FindingParser.parse_findings(malformed_json, sample_view, 1080, 1920)

print("=== STRICT OUTPUT SCHEMA VALIDATION TEST ===")
print(f"[Valid JSON Test]     : {len(findings_valid)} finding(s) extracted, {len(errors_valid)} error(s)")
if findings_valid:
    f = findings_valid[0]
    finding_type = f.type.value if hasattr(f.type, "value") else str(f.type)
    print(f"  -> Type: {finding_type}, Box: {f.bbox_xyxy}, Conf: {f.confidence}")

print(f"\n[Malformed JSON Test] : {len(findings_malformed)} finding(s) extracted, {len(errors_malformed)} error(s)")
for err in errors_malformed:
    print(f"  [X] Caught Validation Fault: {err}")
print("[+] Strict validation guarantees false-alarms and reversed boxes never corrupt test synthesis.")


## 7. Unified Evaluation Engine: Raw Global vs. SAHI Slicing
This engine performs:
1. **Raw Global Inspection:** Scales whole image to max 1280 dimension, performs a single inference forward pass, and parses the verdict.
2. **SAHI Slicing Inspection:** Plans a multi-view pyramid (1 Global Context + 3 Overlapping Tiles), executes slice-level forward passes, maps localized bounding boxes back to original physical screen pixels, and deduplicates overlapping detections with IoU clustering.
3. **Side-by-Side Visual Plot:** Displays Raw Left vs SAHI Right showing exact model detections, bounding boxes, latencies, and explanations without needing hardcoded expected labels.


In [ ]:
# Cell 7: Unified Raw Global vs. SAHI Multi-Slice Slicing Evaluator
def run_single_inference(img, custom_prompt=None):
    """Executes 4-bit VLM forward pass on an image crop."""
    prompt = custom_prompt or (
        "<|im_start|>user\n"
        "<|vision_start|><|image_pad|><|vision_end|>"
        "Analyze this mobile screen for non-crash visual and semantic defects.\n\n"
        "Return the verdict in the specified JSON format.<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    disp = preprocess_view_image(img, max_dim=(720, 1280))
    inputs = processor(text=[prompt], images=[disp], return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=110, do_sample=False)
    raw_response = processor.batch_decode(
        [out[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    return raw_response

def evaluate_screen(image_path, title_prefix="Visual Defect Analysis"):
    """
    Evaluates a screen with both Raw Global Inspection and SAHI Slicing Inspection.
    Renders side-by-side visual comparison showing actual model outputs.
    """
    img_file = resolve_path(image_path)
    if not img_file.exists():
        print(f"[!] File not found: {img_file}")
        return None

    img = Image.open(img_file).convert("RGB")
    orig_w, orig_h = img.size

    # ----------------- 1. RAW GLOBAL INSPECTION -----------------
    t0_global = time.perf_counter()
    raw_global_text = run_single_inference(img)
    lat_global = (time.perf_counter() - t0_global) * 1000.0

    global_view = ViewPlanner.plan_views(orig_w, orig_h, policy=CoveragePolicy.GLOBAL_ONLY)[0]
    findings_global, _ = FindingParser.parse_findings(raw_global_text, global_view, orig_w, orig_h)
    
    raw_has_defect = len(findings_global) > 0
    raw_defect_type = str(findings_global[0].type) if raw_has_defect else "none"
    raw_bbox = findings_global[0].bbox_xyxy if raw_has_defect else None
    raw_conf = findings_global[0].confidence if raw_has_defect else 1.0
    raw_expl = findings_global[0].explanation if raw_has_defect else "Clean screen / no defect detected"

    # ----------------- 2. SAHI SLICING INSPECTION -----------------
    t0_sahi = time.perf_counter()
    sahi_views = ViewPlanner.plan_views(orig_w, orig_h, policy=CoveragePolicy.GLOBAL_PLUS_TILES_V1)
    all_sahi_findings = []
    
    for view in sahi_views:
        cx, cy, cw, ch = view.crop_rect
        view_crop = img if view.is_global else img.crop((cx, cy, cx+cw, cy+ch))
        tile_text = run_single_inference(view_crop)
        f_list, _ = FindingParser.parse_findings(tile_text, view, orig_w, orig_h)
        all_sahi_findings.extend(f_list)

    deduped_sahi = OverlapDeduplicator.deduplicate(all_sahi_findings, iou_threshold=0.5)
    lat_sahi = (time.perf_counter() - t0_sahi) * 1000.0

    sahi_has_defect = len(deduped_sahi) > 0
    sahi_defect_type = str(deduped_sahi[0].type) if sahi_has_defect else "none"
    sahi_bbox = deduped_sahi[0].bbox_xyxy if sahi_has_defect else None
    sahi_conf = deduped_sahi[0].confidence if sahi_has_defect else 1.0
    sahi_expl = deduped_sahi[0].explanation if sahi_has_defect else "Clean screen / no defect detected"

    # ----------------- 3. SIDE-BY-SIDE PLOT -----------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), dpi=105)

    # Left Plot: Raw Global
    disp_global = img.copy()
    disp_global.thumbnail((720, 1280), Image.Resampling.LANCZOS)
    scale_x = disp_global.width / orig_w
    scale_y = disp_global.height / orig_h

    ax1.imshow(disp_global)
    ax1.axis("off")
    color_raw = "#e53e3e" if raw_has_defect else "#38a169"
    ax1.set_title(f"1. Raw Global Inspection\nDetected: [{raw_defect_type.upper()}] | {lat_global:.0f}ms", 
                  fontsize=11, fontweight="bold", pad=10, color=color_raw)
    
    if raw_has_defect and raw_bbox:
        rx1, ry1, rx2, ry2 = raw_bbox
        rect1 = patches.Rectangle((rx1*scale_x, ry1*scale_y), (rx2-rx1)*scale_x, (ry2-ry1)*scale_y,
                                  linewidth=3.5, edgecolor="red", facecolor="none")
        ax1.add_patch(rect1)
        ax1.text(rx1*scale_x + 4, max(25, ry1*scale_y - 8), f"{raw_defect_type.upper()} ({raw_conf:.2f})",
                 fontsize=9, fontweight="bold", color="white",
                 bbox=dict(facecolor="#e53e3e", edgecolor="none", boxstyle="round,pad=0.3", alpha=0.9))

    # Right Plot: SAHI Multi-Slice
    disp_sahi = img.copy()
    disp_sahi.thumbnail((720, 1280), Image.Resampling.LANCZOS)
    ax2.imshow(disp_sahi)
    ax2.axis("off")
    color_sahi = "#2b6cb0" if sahi_has_defect else "#38a169"
    ax2.set_title(f"2. SAHI Multi-Slice Slicing\nDetected: [{sahi_defect_type.upper()}] ({len(sahi_views)} views) | {lat_sahi:.0f}ms", 
                  fontsize=11, fontweight="bold", pad=10, color=color_sahi)

    if sahi_has_defect and sahi_bbox:
        sx1, sy1, sx2, sy2 = sahi_bbox
        rect2 = patches.Rectangle((sx1*scale_x, sy1*scale_y), (sx2-sx1)*scale_x, (sy2-sy1)*scale_y,
                                  linewidth=3.5, edgecolor="#3182ce", facecolor="none")
        ax2.add_patch(rect2)
        ax2.text(sx1*scale_x + 4, max(25, sy1*scale_y - 8), f"{sahi_defect_type.upper()} ({sahi_conf:.2f})",
                 fontsize=9, fontweight="bold", color="white",
                 bbox=dict(facecolor="#3182ce", edgecolor="none", boxstyle="round,pad=0.3", alpha=0.9))

    plt.suptitle(f"{title_prefix}\nFile: {img_file.name} ({orig_w}x{orig_h}px)", fontsize=12, fontweight="bold")
    plt.figtext(0.5, 0.01, f"Raw Explanation : {raw_expl}\nSAHI Explanation: {sahi_expl}", ha="center", fontsize=8.5,
                 bbox=dict(facecolor="#f7fafc", edgecolor="#cbd5e0", boxstyle="round,pad=0.5"))
    plt.tight_layout()
    plt.show()

    return {
        "screen": img_file.name,
        "raw_detected": raw_defect_type,
        "raw_has_defect": raw_has_defect,
        "raw_latency_ms": round(lat_global, 1),
        "sahi_detected": sahi_defect_type,
        "sahi_has_defect": sahi_has_defect,
        "sahi_views_count": len(sahi_views),
        "sahi_latency_ms": round(lat_sahi, 1),
        "sahi_findings_count": len(deduped_sahi),
        "explanation": sahi_expl
    }


## 8. Interactive Verification on Real Bug Screens
We evaluate the Oracle on genuine mobile bug screens from external benchmarks and mutated APK sessions:
1. **Test 1:** OwlEye Benchmark Defect (`dataset/bugs/bug.4001.jpg`)
2. **Test 2:** OwlEye Real Screen Defect (`dataset/bugs/bug.4552.jpg`)
3. **Test 3:** Real Injected Component Overlap (`dataset/mutated_bugs/10036_mut_overlap.jpg`)
4. **Test 4:** Real Text Truncation Defect (`dataset/mutated_bugs/10040_mut_trunc.jpg`)
5. **Test 5:** Real Missing Component Defect (`dataset/mutated_bugs/10053_mut_missing.jpg`)
6. **Test 6:** Clean Mobile UI Screen Baseline (`dataset/clean/100.jpg`)
7. **Test 7:** High-Aspect Viewport Defect (`opening_screen.png`)


In [ ]:
# Test 1: Pure OwlEye Benchmark Defect
res1 = evaluate_screen(
    "dataset/bugs/bug.4001.jpg",
    title_prefix="Test 1: OwlEye Benchmark Bug (dataset/bugs/bug.4001.jpg)"
)


In [ ]:
# Test 2: OwlEye Real Screen Defect
res2 = evaluate_screen(
    "dataset/bugs/bug.4552.jpg",
    title_prefix="Test 2: OwlEye Real Screen Bug (dataset/bugs/bug.4552.jpg)"
)


In [ ]:
# Test 3: Rico Mutated Component Overlap
res3 = evaluate_screen(
    "dataset/mutated_bugs/10036_mut_overlap.jpg",
    title_prefix="Test 3: Injected Component Collision (10036_mut_overlap.jpg)"
)


In [ ]:
# Test 4: Real Text Truncation Defect
res4 = evaluate_screen(
    "dataset/mutated_bugs/10040_mut_trunc.jpg",
    title_prefix="Test 4: Text Truncation Defect (10040_mut_trunc.jpg)"
)


In [ ]:
# Test 5: Real Missing Component Defect
res5 = evaluate_screen(
    "dataset/mutated_bugs/10053_mut_missing.jpg",
    title_prefix="Test 5: Missing Component Defect (10053_mut_missing.jpg)"
)


In [ ]:
# Test 6: Clean Rico Screen Baseline
res6 = evaluate_screen(
    "dataset/clean/100.jpg",
    title_prefix="Test 6: Clean Screen Baseline (dataset/clean/100.jpg)"
)


In [ ]:
# Test 7: High-Aspect-Ratio Viewport Defect
res7 = evaluate_screen(
    "opening_screen.png",
    title_prefix="Test 7: High-Aspect-Ratio Viewport Defect (opening_screen.png)"
)


## 9. Performance Comparison Summary (Raw Global vs. SAHI Slicing)
Consolidates results across all evaluated benchmark screens into a structured summary table.


In [ ]:
# Cell 8: Generate Comparison Table
results = [r for r in [res1, res2, res3, res4, res5, res6, res7] if r is not None]
df = pd.DataFrame(results)

# Display formatted table
cols = ["screen", "raw_detected", "raw_latency_ms", "sahi_detected", "sahi_views_count", "sahi_latency_ms", "sahi_findings_count"]
display_cols = [c for c in cols if c in df.columns]
print("=== ORACLE EVALUATION SUMMARY: RAW GLOBAL VS SAHI SLICING ===")
try:
    from IPython.display import display
    display(df[display_cols])
except Exception:
    print(df[display_cols].to_string())

# Latency and detection observations
avg_raw_lat = df["raw_latency_ms"].mean() if "raw_latency_ms" in df else 0
avg_sahi_lat = df["sahi_latency_ms"].mean() if "sahi_latency_ms" in df else 0
print(f"\n[*] Average Raw Latency  : {avg_raw_lat:.1f} ms")
print(f"[*] Average SAHI Latency : {avg_sahi_lat:.1f} ms (4 views analyzed per screen)")


## 10. Contract V2 & Component 4 Test Generation Handover
Component 3 packages the final verdict into a strongly typed `VerdictEnvelopeV2`.
- When a defect is localized, it sets `handover_outcome: "ready_for_test_generation"` for Appium test script synthesis in C4.
- When visual coverage is partial or inconclusive, it preserves `has_gui_defect: null` and sets `handover_outcome: "evidence_only"`, preventing test flakiness.


In [ ]:
# Cell 9: Contract V2 Serialization & C4 Handover
from vision_oracle.contracts.v2 import (
    VerdictEnvelopeV2,
    AnalysisDecision,
    AnalysisStatus,
    Coverage,
    CoveragePolicy,
    ScreenshotRef,
    Finding,
    FindingType
)

# Build verified Contract V2 envelope
sample_finding = Finding(
    finding_id="find_demo_01",
    type=FindingType.COMPONENT_OVERLAP,
    bbox_xyxy=[120, 940, 980, 1140],
    explanation="Button occludes edit text input box in payment section.",
    source="vlm",
    confidence=0.91,
    confidence_method="uncalibrated",
    view_ids=["tile_1"],
)

envelope = VerdictEnvelopeV2(
    verdict_id="verdict_demo_sahi_01",
    job_id="job_demo_sahi",
    session_id="session_live_test",
    capture_id="cap_live_01",
    action_id="act_checkout_tap",
    decision=AnalysisDecision.DEFECT_DETECTED,
    analysis_status=AnalysisStatus.COMPLETE,
    model_run_id="qwen2-vl-2b-c3-lora-sahi",
    coverage=Coverage(policy=CoveragePolicy.GLOBAL_PLUS_TILES_V1, required_views_total=4, required_views_analyzed=4),
    screenshot=ScreenshotRef(path="dataset/mutated_bugs/10036_mut_overlap.jpg", sha256="abc123sha", width=1080, height=1920),
    primary_finding=sample_finding,
    findings=[sample_finding],
    timings={"preprocess_ms": 12.4, "inference_ms": 1420.0, "total_pipeline_ms": 1450.2},
    metadata={"pipeline": "c3_sahi_multimodal", "device_gpu": "RTX 3050 Laptop GPU"}
)

c4_handover = envelope.to_c4_verdict()
c4_payload = c4_handover.model_dump() if hasattr(c4_handover, "model_dump") else (c4_handover.dict() if hasattr(c4_handover, "dict") else c4_handover)

print("=== CONTRACT V2 COMPONENT 4 HANDOVER PAYLOAD ===")
print(json.dumps(c4_payload, indent=2, default=str))
handover_outcome = c4_payload.get("evidence", {}).get("handover_outcome", "ready_for_test_generation") if isinstance(c4_payload, dict) else "ready"
print(f"\n[+] Handover Outcome: '{handover_outcome}'")
print("[+] Ready for downstream Appium test synthesis in Component 4.")
